# 01 · Completeness and grain

| Notebook | What it answers |
|---|---|
| **01** | §0 what the publisher documents · §1 did every row load, which period each file covers, and how the source changes between downloads · §2 what one row is and which columns identify it |

**Before running:** run the pipeline once (`python -m pipeline.run`) so `data/warehouse.duckdb` exists, and `python -m pipeline.ckan` for the column notes used in §0.

In [46]:
from helpers import connect, show, profile, by_year, values_by_year, DATA_DIR

con, years = connect()
print(f"years: {years[0]}-{years[-1]} ({len(years)} files)")
show("SELECT count(*) AS total_rows FROM s")


show("SELECT * from s LIMIT 10")

years: 2012-2026 (15 files)
┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│     506977 │
└────────────┘

┌───────────┬──────────┬──────────────────┬───────────────────────────────────────┬──────────────────┬──────────────────┬───┬─────────┬──────────┬──────────┬──────────┬──────────────────────┬──────────┬─────────────────┐
│ file_year │   _id    │      PARID       │             FULL_ADDRESS              │ PROPERTYHOUSENUM │ PROPERTYFRACTION │ … │  PRICE  │ DEEDBOOK │ DEEDPAGE │ SALECODE │       SALEDESC       │ INSTRTYP │  INSTRTYPDESC   │
│   int32   │ varchar  │     varchar      │                varchar                │     varchar      │     varchar      │ … │ varchar │ varchar  │ varchar  │ varchar  │       varchar        │ varchar  │     varchar     │
├───────────┼──────────┼──────────────────┼───────────────────────────────────────┼──────────────────┼──────────────────┼───┼─────────┼──────────┼──────────┼──────────┼──────────────────────┼──────────┼────────────────

## 0. Source and context

**Question:** what does the publisher say about each column, and is that description there for every year's file?

In [47]:
# The column labels/notes as the source publishes them for each year (written by pipeline/ckan.py)
dictionary = (DATA_DIR / "data_dictionary.csv").as_posix()
show(f"""
SELECT "column", count(*) AS years_listed,
       count(*) FILTER (WHERE notes IS NULL OR trim(notes) = '') AS years_without_notes,
       any_value(label) AS label
FROM read_csv('{dictionary}', all_varchar=true)
GROUP BY 1 ORDER BY 1
""", width=260)

┌─────────────────────────┬──────────────┬─────────────────────┬───────────────────────────────────────────┐
│         column          │ years_listed │ years_without_notes │                   label                   │
│         varchar         │    int64     │        int64        │                  varchar                  │
├─────────────────────────┼──────────────┼─────────────────────┼───────────────────────────────────────────┤
│ DEEDBOOK                │           15 │                   1 │ Deed Book Number                          │
│ DEEDPAGE                │           15 │                   1 │ Page Number                               │
│ FULL_ADDRESS            │           15 │                   1 │ Full address                              │
│ INSTRTYP                │           15 │                   1 │ Instrument Type                           │
│ INSTRTYPDESC            │           15 │                   1 │ Instrument Description                    │
│ MUNICODE         

In [48]:
show(f"""
SELECT year, count(*) AS columns,
       count(*) FILTER (WHERE notes IS NULL OR trim(notes) = '') AS columns_without_notes
FROM read_csv('{dictionary}', all_varchar=true)
GROUP BY 1 ORDER BY 1
""")

┌─────────┬─────────┬───────────────────────┐
│  year   │ columns │ columns_without_notes │
│ varchar │  int64  │         int64         │
├─────────┼─────────┼───────────────────────┤
│ 2012    │      25 │                     0 │
│ 2013    │      25 │                     0 │
│ 2014    │      25 │                     0 │
│ 2015    │      25 │                     0 │
│ 2016    │      25 │                     0 │
│ 2017    │      25 │                     0 │
│ 2018    │      25 │                     0 │
│ 2019    │      25 │                     0 │
│ 2020    │      25 │                     0 │
│ 2021    │      25 │                     0 │
│ 2022    │      25 │                     0 │
│ 2023    │      25 │                     0 │
│ 2024    │      25 │                     0 │
│ 2025    │      25 │                     0 │
│ 2026    │      25 │                    25 │
└─────────┴─────────┴───────────────────────┘
  15 rows                         3 columns



**What this shows:** the 2026 data dictionary has no description for any of its 25 columns, while 2012 to 2025 describe all 25. All 25 columns are listed in all 15 years, so I assume the earlier descriptions apply to 2026 too.

## 1. Completeness

### 1.0 Can the pipeline's own records be trusted?

**Question:** can the pipeline's own records be trusted? Later checks use the `meta.*` tables as evidence, so first I check them against what's actually on disk and in the tables.

In [49]:
# For every recorded raw file: does it still exist, and does its SHA-256 still match?
# load_manifest = files that were loaded; landing_alias = re-downloads with identical rows (kept, not reloaded)

import pyarrow as pa
from pipeline.ingest import file_sha256, resolve_path

recorded = con.execute("""
SELECT 'load_manifest' AS recorded_in, year, version, landing_file, file_sha256 FROM meta.load_manifest
UNION ALL
SELECT 'landing_alias', year, version, landing_file, file_sha256 FROM meta.landing_alias
""").fetchall()
rows = []
for recorded_in, year, version, landing_file, sha in recorded:
    exists = resolve_path(landing_file).is_file()
    rows.append((recorded_in, year, version, exists, exists and file_sha256(landing_file) == sha))
landing_check = pa.table(dict(zip(["recorded_in", "year", "version", "file_exists", "sha256_matches"], map(list, zip(*rows)))))
con.register("landing_check", landing_check)
show("""
SELECT recorded_in, count(*) AS files, count(*) FILTER (WHERE NOT file_exists) AS missing,
       count(*) FILTER (WHERE file_exists AND NOT sha256_matches) AS sha256_mismatch
FROM landing_check GROUP BY 1 ORDER BY 1
""")

┌───────────────┬───────┬─────────┬─────────────────┐
│  recorded_in  │ files │ missing │ sha256_mismatch │
│    varchar    │ int64 │  int64  │      int64      │
├───────────────┼───────┼─────────┼─────────────────┤
│ landing_alias │    35 │       0 │               0 │
│ load_manifest │    25 │       0 │               0 │
└───────────────┴───────┴─────────┴─────────────────┘



`files` = how many raw files the pipeline recorded, `missing` = how many are no longer on disk, `sha256_mismatch` = how many are on disk but their contents changed.
- `load_manifest`, 25 files: files loaded as a version. Each of the 15 years was loaded once at the start (15), and each of the 10 real changes added a version (10).
- `landing_alias`, 35 files: re-downloads with exactly the same rows as a version I already have (the source re-published them with new `_id` values). Kept as evidence, not loaded again.

In [50]:
# For each year: rows recorded at load time vs rows in the table now

show("SELECT *, manifest_rows = table_rows AS match FROM (" + " UNION ALL ".join(
    f"SELECT {y} AS year, (SELECT m.loaded_row_count FROM meta.load_manifest m JOIN meta.active_version a USING (year, version) "
    f"WHERE m.year = {y}) AS manifest_rows, (SELECT count(*) FROM bronze.sales_{y}) AS table_rows"
    for y in years) + ") ORDER BY year")

┌───────┬───────────────┬────────────┬─────────┐
│ year  │ manifest_rows │ table_rows │  match  │
│ int32 │     int64     │   int64    │ boolean │
├───────┼───────────────┼────────────┼─────────┤
│  2012 │         33651 │      33651 │ true    │
│  2013 │         34095 │      34095 │ true    │
│  2014 │         33670 │      33670 │ true    │
│  2015 │         34686 │      34686 │ true    │
│  2016 │         36011 │      36011 │ true    │
│  2017 │         35282 │      35282 │ true    │
│  2018 │         36488 │      36488 │ true    │
│  2019 │         35671 │      35671 │ true    │
│  2020 │         35348 │      35348 │ true    │
│  2021 │         40234 │      40234 │ true    │
│  2022 │         34827 │      34827 │ true    │
│  2023 │         30862 │      30862 │ true    │
│  2024 │         31115 │      31115 │ true    │
│  2025 │         31555 │      31555 │ true    │
│  2026 │         23482 │      23482 │ true    │
└───────┴───────────────┴────────────┴─────────┘
  15 rows           

`manifest_rows` = the row count the pipeline wrote down when it loaded the version in use. `table_rows` = the count in the table today.

**What this shows:** `manifest_rows` equals `table_rows` for all 15 years, and the query before found every recorded raw file on disk with the same SHA-256. So the `meta.*` tables can be used as evidence in the rest of this notebook.

### 1.1 Row counts per table vs the source

**Question:** does each table have as many rows as the source says it should?

In [51]:
show(" UNION ALL ".join(
    f"SELECT '{y}' AS year, 'bronze.sales_{y}' AS table_name, count(*) AS rows FROM bronze.sales_{y}"
    for y in years) + " UNION ALL SELECT 'TOTAL', '', count(*) FROM s")

┌─────────┬───────────────────┬────────┐
│  year   │    table_name     │  rows  │
│ varchar │      varchar      │ int64  │
├─────────┼───────────────────┼────────┤
│ 2012    │ bronze.sales_2012 │  33651 │
│ 2013    │ bronze.sales_2013 │  34095 │
│ 2014    │ bronze.sales_2014 │  33670 │
│ 2015    │ bronze.sales_2015 │  34686 │
│ 2016    │ bronze.sales_2016 │  36011 │
│ 2017    │ bronze.sales_2017 │  35282 │
│ 2018    │ bronze.sales_2018 │  36488 │
│ 2019    │ bronze.sales_2019 │  35671 │
│ 2020    │ bronze.sales_2020 │  35348 │
│ 2021    │ bronze.sales_2021 │  40234 │
│ 2022    │ bronze.sales_2022 │  34827 │
│ 2023    │ bronze.sales_2023 │  30862 │
│ 2024    │ bronze.sales_2024 │  31115 │
│ 2025    │ bronze.sales_2025 │  31555 │
│ 2026    │ bronze.sales_2026 │  23482 │
│ TOTAL   │                   │ 506977 │
└─────────┴───────────────────┴────────┘
  16 rows                    3 columns



**What this shows:** the 15 tables hold 506,977 rows in total. This query only counts what was loaded; the next query compares it with what WPRDC reports.

**Question:** does the source's own row count match what was loaded? (latest run: `source_rows` = what WPRDC's API reported for the file, `loaded_rows` = what the pipeline loaded)

In [52]:
show("""
SELECT year, change_type, ckan_row_count AS source_rows, loaded_row_count AS loaded_rows,
       ckan_row_count = loaded_row_count AS match
FROM meta.run_log
WHERE run_id = (SELECT run_id FROM meta.run_log WHERE change_type IS NOT NULL ORDER BY started_at DESC LIMIT 1)
ORDER BY year
""")

┌───────┬──────────────────────────┬─────────────┬─────────────┬─────────┐
│ year  │       change_type        │ source_rows │ loaded_rows │  match  │
│ int32 │         varchar          │    int64    │    int64    │ boolean │
├───────┼──────────────────────────┼─────────────┼─────────────┼─────────┤
│  2012 │ identical_file           │       33651 │       33651 │ true    │
│  2013 │ identical_file           │       34095 │       34095 │ true    │
│  2014 │ identical_file           │       33670 │       33670 │ true    │
│  2015 │ identical_file           │       34686 │       34686 │ true    │
│  2016 │ identical_file           │       36011 │       36011 │ true    │
│  2017 │ identical_file           │       35282 │       35282 │ true    │
│  2018 │ identical_file           │       36488 │       36488 │ true    │
│  2019 │ identical_file           │       35671 │       35671 │ true    │
│  2020 │ republished_same_content │       35348 │       35348 │ true    │
│  2021 │ republished_sam

**What this shows:** in the latest run the two counts are equal for all 15 years, so no rows were lost or added while downloading and loading.

### 1.2 Which period each file covers

**Question:** which sale dates does each file contain? Do files overlap, and is any period incomplete?

In [53]:
show("""
SELECT file_year,
count(*) AS rows, 
min(SALEDATE) AS first_sale, 
max(SALEDATE) AS last_sale FROM s 
GROUP BY 1 
ORDER BY 1
""")

┌───────────┬───────┬────────────┬────────────┐
│ file_year │ rows  │ first_sale │ last_sale  │
│   int32   │ int64 │  varchar   │  varchar   │
├───────────┼───────┼────────────┼────────────┤
│      2012 │ 33651 │ 2012-01-01 │ 2012-12-31 │
│      2013 │ 34095 │ 2013-01-01 │ 2013-12-31 │
│      2014 │ 33670 │ 2014-01-01 │ 2014-12-31 │
│      2015 │ 34686 │ 2015-01-01 │ 2015-12-31 │
│      2016 │ 36011 │ 2016-01-01 │ 2016-12-31 │
│      2017 │ 35282 │ 2017-01-01 │ 2017-12-31 │
│      2018 │ 36488 │ 2018-01-01 │ 2018-12-31 │
│      2019 │ 35671 │ 2019-01-01 │ 2019-12-31 │
│      2020 │ 35348 │ 2020-01-01 │ 2020-12-31 │
│      2021 │ 40234 │ 2021-01-01 │ 2021-12-31 │
│      2022 │ 34827 │ 2022-01-01 │ 2022-12-31 │
│      2023 │ 30862 │ 2023-01-01 │ 2023-12-31 │
│      2024 │ 31115 │ 2024-01-01 │ 2024-12-31 │
│      2025 │ 31555 │ 2025-01-01 │ 2025-12-31 │
│      2026 │ 23482 │ 2026-01-01 │ 2026-10-22 │
└───────────┴───────┴────────────┴────────────┘
  15 rows                           4 co

In [54]:
# Rows whose sale year differs from the year of the file they are in (unparsed = not yyyy-mm-dd)
show("""
SELECT file_year,
       count(*) FILTER (WHERE year(try_strptime(SALEDATE, '%Y-%m-%d')) <> file_year) AS sale_year_not_file_year,
       count(*) FILTER (WHERE try_strptime(SALEDATE, '%Y-%m-%d') IS NULL)            AS sale_date_unparsed
FROM s GROUP BY 1 ORDER BY 1
""")

┌───────────┬─────────────────────────┬────────────────────┐
│ file_year │ sale_year_not_file_year │ sale_date_unparsed │
│   int32   │          int64          │       int64        │
├───────────┼─────────────────────────┼────────────────────┤
│      2012 │                       0 │                  0 │
│      2013 │                       0 │                  0 │
│      2014 │                       0 │                  0 │
│      2015 │                       0 │                  0 │
│      2016 │                       0 │                  0 │
│      2017 │                       0 │                  0 │
│      2018 │                       0 │                  0 │
│      2019 │                       0 │                  0 │
│      2020 │                       0 │                  0 │
│      2021 │                       0 │                  0 │
│      2022 │                       0 │                  0 │
│      2023 │                       0 │                  0 │
│      2024 │           

In [55]:
# The newest file: how far does it reach, compared with today?
show("""
SELECT file_year, max(SALEDATE) AS last_sale, current_date AS today
FROM s WHERE file_year = (SELECT max(file_year) FROM s) GROUP BY 1
""")

┌───────────┬────────────┬────────────┐
│ file_year │ last_sale  │   today    │
│   int32   │  varchar   │    date    │
├───────────┼────────────┼────────────┤
│      2026 │ 2026-10-22 │ 2026-11-01 │
└───────────┴────────────┴────────────┘



**What this shows:** every row's sale date is in its file's year (0 rows from another year, 0 unreadable dates), so each file holds exactly one sale year. Each file runs from 1 Jan to 31 Dec, except 2026, which isn't finished: its last sale is 2026-10-22, and the file was last modified on 2026-10-29.

So no two files share a sale year.

### 1.3 How the source changes between downloads

**Question:** what happens to the source between runs: same file, re-published with the same rows, or really changed?

In [56]:
# rows_with_new_id = rows whose _id differs from the same row in the version we hold (rows paired on every other column).
# ids_not_in_old   = _id values in the new file that don't exist anywhere in the version we hold.
# Both are measured only for republished_same_content; NULL otherwise 
# content_changed/new: rows_added/rows_removed.
show("""
SELECT strftime(min(started_at), '%Y-%m-%d %H:%M') AS run_started, change_type,
       count(*) AS years, sum(rows_with_new_id) AS rows_with_new_id, sum(ids_not_in_old) AS ids_not_in_old,
       sum(loaded_row_count) AS rows_in_those_years
FROM meta.run_log WHERE change_type IS NOT NULL
GROUP BY run_id, change_type ORDER BY run_started, change_type
""")

┌──────────────────┬──────────────────────────┬───────┬──────────────────┬────────────────┬─────────────────────┐
│   run_started    │       change_type        │ years │ rows_with_new_id │ ids_not_in_old │ rows_in_those_years │
│     varchar      │         varchar          │ int64 │      int128      │     int128     │       int128        │
├──────────────────┼──────────────────────────┼───────┼──────────────────┼────────────────┼─────────────────────┤
│ 2026-10-27 15:55 │ identical_file           │     3 │             NULL │           NULL │               93649 │
│ 2026-10-27 15:55 │ republished_same_content │    12 │           412905 │         412905 │              412905 │
│ 2026-10-29 12:46 │ content_changed          │     4 │             NULL │           NULL │              121060 │
│ 2026-10-29 12:46 │ republished_same_content │    11 │           385795 │         385795 │              385795 │
│ 2026-10-29 16:45 │ failed                   │     1 │             NULL │           NUL

`years` = how many of the 15 files got that result in that run. `rows_with_new_id` and `ids_not_in_old` are only filled for `republished_same_content`: the first counts unchanged rows that got a new `_id`, the second counts `_id` values that didn't exist at all in the version I have. `rows_in_those_years` = total rows of those files.

**What this shows:**
- **35 times the file was re-published with the same rows** (`republished_same_content`). Each time, `rows_with_new_id` equals the row count: the rows are the same, but every row got a new `_id`. So the source rebuilds and renumbers files even when nothing changed.
- **31 times the file was exactly the same** (`identical_file`).
- **7 times the file really changed** (`content_changed`). `meta.change_log` (below) has 10 changes: the first 3 come from runs before the pipeline recorded `change_type`, and this query leaves those out (`WHERE change_type IS NOT NULL`).
- **2 downloads were refused** (`failed`), explained at the end of this section.

In [57]:
# Latest run, per year
show("""
SELECT year, change_type, rows_with_new_id, ids_not_in_old, loaded_row_count, rows_added, rows_removed, source_last_modified
FROM meta.run_log
WHERE run_id = (SELECT run_id FROM meta.run_log WHERE change_type IS NOT NULL ORDER BY started_at DESC LIMIT 1)
ORDER BY year
""")

┌───────┬──────────────────────────┬──────────────────┬────────────────┬──────────────────┬────────────┬──────────────┬────────────────────────────┐
│ year  │       change_type        │ rows_with_new_id │ ids_not_in_old │ loaded_row_count │ rows_added │ rows_removed │    source_last_modified    │
│ int32 │         varchar          │      int64       │     int64      │      int64       │   int64    │    int64     │          varchar           │
├───────┼──────────────────────────┼──────────────────┼────────────────┼──────────────────┼────────────┼──────────────┼────────────────────────────┤
│  2012 │ identical_file           │             NULL │           NULL │            33651 │       NULL │         NULL │ 2026-10-29T07:03:18.327231 │
│  2013 │ identical_file           │             NULL │           NULL │            34095 │       NULL │         NULL │ 2026-10-29T07:06:27.265619 │
│  2014 │ identical_file           │             NULL │           NULL │            33670 │       NULL │  

**Question:** when does the source update each file? (`last_modified` as reported by the source at each download)

In [58]:
show("""
SELECT year, count(DISTINCT last_modified) AS distinct_values, list(DISTINCT last_modified ORDER BY last_modified) AS last_modified_seen
FROM (SELECT year, source_last_modified AS last_modified FROM meta.run_log WHERE source_last_modified IS NOT NULL
      UNION ALL
      SELECT year, source_last_modified FROM meta.load_manifest WHERE source_last_modified IS NOT NULL)
GROUP BY 1 ORDER BY 1
""", width=300)

┌───────┬─────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ year  │ distinct_values │                                                    last_modified_seen                                                    │
│ int32 │      int64      │                                                        varchar[]                                                         │
├───────┼─────────────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  2012 │               4 │ ['2026-10-25T07:51:40.711082', '2026-10-26T07:12:42.824834', '2026-10-28T07:02:42.214475', '2026-10-29T07:03:18.327231'] │
│  2013 │               4 │ ['2026-10-24T07:32:18.049305', '2026-10-26T07:16:36.757530', '2026-10-28T07:05:43.846852', '2026-10-29T07:06:27.265619'] │
│  2014 │               4 │ ['2026-10-24T07:39:21.768386', '2026-10-26T07:22:03.833720', '2026

`last_modified` = the time the source says it last updated the file, as seen at each download.

**What this shows:** every day I downloaded, every file had a new `last_modified`, always between 07:02 and 08:21. Several downloads on the same day (29 Oct) saw the same value. So the source seems to rebuild every file early each morning, even when the rows don't change. I didn't download every day, so this is an assumption from the timestamps, to be confirmed with the publisher.

**Question:** which real changes were found, and how big were they?

In [59]:
show("""
SELECT year, from_version, to_version, rows_before, rows_after, rows_added, rows_removed, diff_table,
       strftime(detected_at, '%Y-%m-%d %H:%M') AS detected
FROM meta.change_log ORDER BY detected_at, year
""")

┌───────┬──────────────┬────────────┬─────────────┬────────────┬────────────┬──────────────┬──────────────────────┬──────────────────┐
│ year  │ from_version │ to_version │ rows_before │ rows_after │ rows_added │ rows_removed │      diff_table      │     detected     │
│ int32 │    int32     │   int32    │    int64    │   int64    │   int64    │    int64     │       varchar        │     varchar      │
├───────┼──────────────┼────────────┼─────────────┼────────────┼────────────┼──────────────┼──────────────────────┼──────────────────┤
│  2013 │            1 │          2 │       33914 │      34095 │        181 │            0 │ meta.diff_2013_v1_v2 │ 2026-10-27 13:34 │
│  2018 │            1 │          2 │       36110 │      36488 │        378 │            0 │ meta.diff_2018_v1_v2 │ 2026-10-27 13:35 │
│  2026 │            1 │          2 │       22955 │      23066 │        139 │           28 │ meta.diff_2026_v1_v2 │ 2026-10-27 13:36 │
│  2017 │            1 │          2 │       35281 │    

One row per real change: the year, the version it went from and to, the row counts before and after, and the rows added and removed (the exact rows are in `diff_table`).

**What this shows:** 10 real changes across 7 year files (2013, 2017, 2018, 2022, 2023, 2025, 2026). Past years change too, not only the current one: 2013 gained 181 rows and 2018 gained 378. 2026 changed three times (+139/−28, +502/−206, +289/−169).

That's why silver is rebuilt from bronze on every run.

**Open question:** how often does WPRDC refresh each file, and how far back can past years change?

**Question:** which downloads did the pipeline refuse, and why?

In [60]:
show("""
SELECT strftime(started_at, '%Y-%m-%d %H:%M') AS run_started, year, message
FROM meta.run_log WHERE status = 'failed' ORDER BY started_at, year
""", width=300)

┌──────────────────┬───────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│   run_started    │ year  │                                                                                message                                                                                │
│     varchar      │ int32 │                                                                                varchar                                                                                │
├──────────────────┼───────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 2026-10-29 16:45 │  2016 │ 2016: source changed during download (last_modified 2026-10-28T07:15:10.324402 -> 2026-10-28T07:15:10.324402, rows 35976 -> 35992)                                    │
│ 2026-10-29 16

**What this shows:** two downloads were refused, both because the source was in the middle of rewriting the file.

The pipeline refuses a download if its row count differs from the one already recorded for the same `last_modified`. The year keeps its current version and is tried again next run. That's what happened here: both years loaded normally in the latest run. The pipeline should run after the source's morning rebuild has finished.

## 2. Structure and grain

### 2.1 What a raw row contains

**Question:** what does a raw row contain, and what types do the columns have?

In [61]:
show("SELECT * FROM s USING SAMPLE 5 ROWS", width=400)

┌───────────┬──────────┬──────────────────┬──────────────────────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────────┬────────────────────┬──────────────────────┬────────────────┬──────────────┬───┬────────────────┬──────────┬─────────────────┬────────────┬────────────┬─────────┬──────────┬──────────┬──────────┬──────────────────────┬──────────┬──────────────────┐
│ file_year │   _id    │      PARID       │       FULL_ADDRESS       │ PROPERTYHOUSENUM │ PROPERTYFRACTION │ PROPERTYADDRESSDIR │ PROPERTYADDRESSSTRE… │ PROPERTYADDRESSSUF │ PROPERTYADDRESSUNIT… │ PROPERTYUNITNO │ PROPERTYCITY │ … │   SCHOOLDESC   │ MUNICODE │    MUNIDESC     │ RECORDDATE │  SALEDATE  │  PRICE  │ DEEDBOOK │ DEEDPAGE │ SALECODE │       SALEDESC       │ INSTRTYP │   INSTRTYPDESC   │
│   int32   │ varchar  │     varchar      │         varchar          │     varchar      │     varchar      │      varchar       │       varchar        │      varchar       │       varchar        │  

In [62]:
show(f"SELECT column_name, column_type FROM (DESCRIBE bronze.sales_{years[-1]})")

┌─────────────────────────┬─────────────┐
│       column_name       │ column_type │
│         varchar         │   varchar   │
├─────────────────────────┼─────────────┤
│ _id                     │ VARCHAR     │
│ PARID                   │ VARCHAR     │
│ FULL_ADDRESS            │ VARCHAR     │
│ PROPERTYHOUSENUM        │ VARCHAR     │
│ PROPERTYFRACTION        │ VARCHAR     │
│ PROPERTYADDRESSDIR      │ VARCHAR     │
│ PROPERTYADDRESSSTREET   │ VARCHAR     │
│ PROPERTYADDRESSSUF      │ VARCHAR     │
│ PROPERTYADDRESSUNITDESC │ VARCHAR     │
│ PROPERTYUNITNO          │ VARCHAR     │
│ PROPERTYCITY            │ VARCHAR     │
│ PROPERTYSTATE           │ VARCHAR     │
│ PROPERTYZIP             │ VARCHAR     │
│ SCHOOLCODE              │ VARCHAR     │
│ SCHOOLDESC              │ VARCHAR     │
│ MUNICODE                │ VARCHAR     │
│ MUNIDESC                │ VARCHAR     │
│ RECORDDATE              │ VARCHAR     │
│ SALEDATE                │ VARCHAR     │
│ PRICE                   │ VARCHA

**What this shows:** every column, including the dates and the price, arrives as text (VARCHAR), and `_id` is the source's row number. The real type of each column is decided in notebook 02.

### 2.2 Is `_id` a key?

**Question:** `_id` is the row number the source gives each row. Is it unique within a file, and across files?

In [63]:
# Within each table: rows vs. distinct _id (duplicate_ids > 0 means _id repeats inside that table)
show(" UNION ALL ".join(
    f"SELECT {y} AS year, count(*) AS rows, count(DISTINCT _id) AS distinct_ids, "
    f"count(*) - count(DISTINCT _id) AS duplicate_ids FROM bronze.sales_{y}"
    for y in years) + " ORDER BY year")

┌───────┬───────┬──────────────┬───────────────┐
│ year  │ rows  │ distinct_ids │ duplicate_ids │
│ int32 │ int64 │    int64     │     int64     │
├───────┼───────┼──────────────┼───────────────┤
│  2012 │ 33651 │        33651 │             0 │
│  2013 │ 34095 │        34095 │             0 │
│  2014 │ 33670 │        33670 │             0 │
│  2015 │ 34686 │        34686 │             0 │
│  2016 │ 36011 │        36011 │             0 │
│  2017 │ 35282 │        35282 │             0 │
│  2018 │ 36488 │        36488 │             0 │
│  2019 │ 35671 │        35671 │             0 │
│  2020 │ 35348 │        35348 │             0 │
│  2021 │ 40234 │        40234 │             0 │
│  2022 │ 34827 │        34827 │             0 │
│  2023 │ 30862 │        30862 │             0 │
│  2024 │ 31115 │        31115 │             0 │
│  2025 │ 31555 │        31555 │             0 │
│  2026 │ 23482 │        23482 │             0 │
└───────┴───────┴──────────────┴───────────────┘
  15 rows           

In [64]:
# Across tables: how many _id values appear in more than one year's table?
show("""
SELECT count(*)                         AS ids_in_multiple_tables,
       sum(tables)                      AS rows_involved,
       max(tables)                      AS max_tables_for_one_id
FROM (SELECT _id, count(DISTINCT file_year) AS tables FROM s GROUP BY _id HAVING count(DISTINCT file_year) > 1)
""")

┌────────────────────────┬───────────────┬───────────────────────┐
│ ids_in_multiple_tables │ rows_involved │ max_tables_for_one_id │
│         int64          │    int128     │         int64         │
├────────────────────────┼───────────────┼───────────────────────┤
│                  22644 │         45288 │                     2 │
└────────────────────────┴───────────────┴───────────────────────┘



In [65]:
# Which pairs of year tables share _id values, and how many?
show("""
SELECT a.file_year AS year_a, b.file_year AS year_b, count(*) AS shared_ids
FROM (SELECT DISTINCT file_year, _id FROM s) a
JOIN (SELECT DISTINCT file_year, _id FROM s) b ON a._id = b._id AND a.file_year < b.file_year
GROUP BY 1, 2 ORDER BY shared_ids DESC
""")

┌────────┬────────┬────────────┐
│ year_a │ year_b │ shared_ids │
│ int32  │ int32  │   int64    │
├────────┼────────┼────────────┤
│   2013 │   2020 │      22644 │
└────────┴────────┴────────────┘



In [66]:
# Look at a few shared _id values side by side: are they the same sale, or unrelated rows?
show("""
SELECT _id, file_year, PARID, SALEDATE, PRICE, SALECODE, FULL_ADDRESS
FROM s
WHERE _id IN (SELECT _id FROM s GROUP BY _id HAVING count(DISTINCT file_year) > 1 LIMIT 5)
ORDER BY _id, file_year
""", width=260)

┌──────────┬───────────┬──────────────────┬────────────┬─────────┬──────────┬──────────────────────────────────────────────┐
│   _id    │ file_year │      PARID       │  SALEDATE  │  PRICE  │ SALECODE │                 FULL_ADDRESS                 │
│ varchar  │   int32   │     varchar      │  varchar   │ varchar │ varchar  │                   varchar                    │
├──────────┼───────────┼──────────────────┼────────────┼─────────┼──────────┼──────────────────────────────────────────────┤
│ 21317847 │      2013 │ 0083F00289000000 │ 2013-05-28 │ 288000  │ 29       │ 604 N SAINT CLAIR ST, PITTSBURGH, PA 15206   │
│ 21317847 │      2020 │ 0014S00013000000 │ 2020-03-03 │ 5000    │ 99       │ 135 LOCUST ST, PITTSBURGH, PA 15210          │
│ 21318204 │      2013 │ 0048B00143000000 │ 2013-05-29 │ 80000   │ X        │ 2153 LOWRIE ST, PITTSBURGH, PA 15212         │
│ 21318204 │      2020 │ 0214A00219000000 │ 2020-10-23 │ 272000  │ 0        │ 328 BREADING AVE, PITTSBURGH, PA 15202       │


**What this shows:** `_id` is unique inside each file (0 duplicates in all 15), but 22,644 `_id` values appear in both the 2013 and the 2020 file, and those rows are completely different records (different parcel, date, price and address). So `_id` is a row number per file, not a sale id. It's also renumbered whenever the source re-publishes a file, so it can't be my key.

**Question:** is file + `_id` unique, and do the `_id` ranges follow the year order?

In [67]:
show("""
SELECT count(*) AS rows,
       count(DISTINCT _id) AS distinct_id,
       count(DISTINCT (file_year, _id)) AS distinct_file_year_id,
       count(DISTINCT PARID) AS distinct_parcels
FROM s
""")

┌────────┬─────────────┬───────────────────────┬──────────────────┐
│  rows  │ distinct_id │ distinct_file_year_id │ distinct_parcels │
│ int64  │    int64    │         int64         │      int64       │
├────────┼─────────────┼───────────────────────┼──────────────────┤
│ 506977 │      484333 │                506977 │           313906 │
└────────┴─────────────┴───────────────────────┴──────────────────┘



In [68]:
# _id range per file
show("SELECT file_year, min(_id::BIGINT) AS min_id, max(_id::BIGINT) AS max_id, count(*) AS rows FROM s GROUP BY 1 ORDER BY 1")

┌───────────┬──────────┬──────────┬───────┐
│ file_year │  min_id  │  max_id  │ rows  │
│   int32   │  int64   │  int64   │ int64 │
├───────────┼──────────┼──────────┼───────┤
│      2012 │ 21215503 │ 21249153 │ 33651 │
│      2013 │ 21306286 │ 21340380 │ 34095 │
│      2014 │ 20892961 │ 20926630 │ 33670 │
│      2015 │ 21497851 │ 21532536 │ 34686 │
│      2016 │ 22203127 │ 22239137 │ 36011 │
│      2017 │ 21734824 │ 21770105 │ 35282 │
│      2018 │ 22302688 │ 22339175 │ 36488 │
│      2019 │ 21682816 │ 21718486 │ 35671 │
│      2020 │ 21317737 │ 21353084 │ 35348 │
│      2021 │ 24240844 │ 24281077 │ 40234 │
│      2022 │ 21029548 │ 21064374 │ 34827 │
│      2023 │ 18533857 │ 18564718 │ 30862 │
│      2024 │ 18413671 │ 18444785 │ 31115 │
│      2025 │ 11698600 │ 11730154 │ 31555 │
│      2026 │  2254297 │  2277778 │ 23482 │
└───────────┴──────────┴──────────┴───────┘
  15 rows                       4 columns



**Question:** does the source keep `_id` the same when it re-publishes a file?

In [69]:
show("""
SELECT strftime(started_at, '%Y-%m-%d') AS run_date, year, loaded_row_count AS rows, rows_with_new_id, ids_not_in_old
FROM meta.run_log WHERE change_type = 'republished_same_content'
ORDER BY started_at, year
""")

┌────────────┬───────┬───────┬──────────────────┬────────────────┐
│  run_date  │ year  │ rows  │ rows_with_new_id │ ids_not_in_old │
│  varchar   │ int32 │ int64 │      int64       │     int64      │
├────────────┼───────┼───────┼──────────────────┼────────────────┤
│ 2026-10-27 │  2012 │ 33651 │            33651 │          33651 │
│ 2026-10-27 │  2014 │ 33670 │            33670 │          33670 │
│ 2026-10-27 │  2015 │ 34686 │            34686 │          34686 │
│ 2026-10-27 │  2016 │ 36011 │            36011 │          36011 │
│ 2026-10-27 │  2017 │ 35281 │            35281 │          35281 │
│ 2026-10-27 │  2019 │ 35671 │            35671 │          35671 │
│ 2026-10-27 │  2020 │ 35348 │            35348 │          35348 │
│ 2026-10-27 │  2021 │ 40234 │            40234 │          40234 │
│ 2026-10-27 │  2022 │ 34826 │            34826 │          34826 │
│ 2026-10-27 │  2023 │ 30861 │            30861 │          30861 │
│ 2026-10-27 │  2024 │ 31115 │            31115 │          311

From the pipeline's records of re-publishes with the same rows: `rows_with_new_id` = rows (matched by content) whose `_id` changed; `ids_not_in_old` = `_id` values not used at all in the version I have.

**What this shows:** file + `_id` is unique for every row today, but `_id` isn't stable: in all 35 re-publishes so far, every row came back with a new `_id`. *Assumption:* the source assigns `_id` whenever it reloads a file, which would also explain why the ranges don't follow the year order. So `_id`, alone or with other columns, only finds a row in the current copy of the file. It can't match the same record across reloads or years, so the key has to come from the data itself.

**Open question:** how does the source assign `_id`?

Whatever the rule is, `_id` can only identify a row in the current copy of a file, because it can change when the file is re-published. So I keep it only for tracing a cleaned row back to its bronze row, not as a key.

### 2.3 Parcels

**Question:** how often does the same parcel (`PARID`) appear, and what does one parcel's history look like?

In [70]:
# total number of parcels that have multiple records
show("""
SELECT COUNT(*) AS total_parcels_with_multiple_records
FROM (
    SELECT PARID
    FROM s
    GROUP BY PARID
    HAVING COUNT(*) > 1
) t
""")

┌─────────────────────────────────────┐
│ total_parcels_with_multiple_records │
│                int64                │
├─────────────────────────────────────┤
│                              131337 │
└─────────────────────────────────────┘



In [71]:
# record count of each parcel that appear in multiple records
show("""
SELECT
    PARID,
    COUNT(*) AS record_count
FROM s
GROUP BY PARID
HAVING COUNT(*) > 1
ORDER BY record_count DESC
LIMIT 10
""")

┌──────────────────┬──────────────┐
│      PARID       │ record_count │
│     varchar      │    int64     │
├──────────────────┼──────────────┤
│ 0431B00017000000 │           24 │
│ 0027D00272000000 │           20 │
│ 0027D00286000000 │           20 │
│ 0027D00263000000 │           20 │
│ 0052A00168000000 │           20 │
│ 0052E00070000000 │           15 │
│ 0039L00335000000 │           15 │
│ 0039L00358000000 │           15 │
│ 0039L00333000000 │           15 │
│ 0052E00071000000 │           15 │
└──────────────────┴──────────────┘
  10 rows               2 columns



In [72]:
# Full history of the most-repeated parcel
show("""
SELECT file_year, SALEDATE, PRICE, SALECODE, SALEDESC, DEEDBOOK, DEEDPAGE FROM s
WHERE PARID = (SELECT PARID FROM s GROUP BY PARID ORDER BY count(*) DESC, PARID LIMIT 1)
ORDER BY SALEDATE
""")

┌───────────┬────────────┬─────────┬──────────┬───────────────────────────────┬──────────┬──────────┐
│ file_year │  SALEDATE  │  PRICE  │ SALECODE │           SALEDESC            │ DEEDBOOK │ DEEDPAGE │
│   int32   │  varchar   │ varchar │ varchar  │            varchar            │ varchar  │ varchar  │
├───────────┼────────────┼─────────┼──────────┼───────────────────────────────┼──────────┼──────────┤
│      2012 │ 2012-02-02 │ 10      │ 9        │ OTHER INVALID SALES INDICATED │ 14803    │ 494      │
│      2012 │ 2012-02-02 │ 10      │ 9        │ OTHER INVALID SALES INDICATED │ 14803    │ 503      │
│      2012 │ 2012-02-02 │ 10      │ 9        │ OTHER INVALID SALES INDICATED │ 14803    │ 516      │
│      2012 │ 2012-02-02 │ 10      │ 9        │ OTHER INVALID SALES INDICATED │ 14803    │ 534      │
│      2012 │ 2012-04-05 │ 10      │ 9        │ OTHER INVALID SALES INDICATED │ 14855    │ 282      │
│      2012 │ 2012-04-05 │ 10      │ 9        │ OTHER INVALID SALES INDICATED │ 14

**Question:** across all parcels: does a parcel repeat inside the same file, across files, or both?

In [73]:
show("""
SELECT count(*) AS parcels,
       count(*) FILTER (WHERE rows > 1)                         AS total_parcels_with_2plus_rows,
       count(*) FILTER (WHERE max_rows_in_one_file > 1)         AS total_parcels_repeated_within_a_file,
       count(*) FILTER (WHERE files > 1)                        AS total_repeated_across_files,
       count(*) FILTER (WHERE max_rows_in_one_file > 1 AND files > 1) AS both
FROM (SELECT PARID, sum(n) AS rows, count(*) AS files, max(n) AS max_rows_in_one_file
      FROM (SELECT PARID, file_year, count(*) AS n FROM s GROUP BY 1, 2) GROUP BY 1)
""")

┌─────────┬───────────────────────────────┬──────────────────────────────────────┬─────────────────────────────┬───────┐
│ parcels │ total_parcels_with_2plus_rows │ total_parcels_repeated_within_a_file │ total_repeated_across_files │ both  │
│  int64  │             int64             │                int64                 │            int64            │ int64 │
├─────────┼───────────────────────────────┼──────────────────────────────────────┼─────────────────────────────┼───────┤
│  313906 │                        131337 │                                29640 │                      118333 │ 16636 │
└─────────┴───────────────────────────────┴──────────────────────────────────────┴─────────────────────────────┴───────┘



**What this shows:** a parcel repeats both within a year and across years. Of 313,906 parcels, 131,337 have more than one row: 29,640 repeat within their yearly file, 118,333 appear in more than one file, and 16,636 do both (29,640 + 118,333 − 16,636 = 131,337). So `PARID` alone isn't a key.

### 2.4 Deeds

**Question:** how many parcels does one deed (book + page) cover?

- **Deed reference** = one `DEEDBOOK` + `DEEDPAGE` pair. The data dictionary says the page is the first page of the deed in the County's deed books, so one reference should point to one document, which can transfer several parcels at once. This section tests whether that always holds.
- **Placeholder** = a book or page that is blank or only zeros (`0`, `00`, …). It doesn't point to a real document.

In [74]:
# How many parcels does one deed (book + page) cover?
show("""
SELECT parcels_on_deed, count(*) AS deeds
FROM (SELECT DEEDBOOK, DEEDPAGE, count(DISTINCT PARID) AS parcels_on_deed
      FROM s WHERE DEEDBOOK <> '' GROUP BY 1, 2)
GROUP BY 1 ORDER BY 1 LIMIT 15
""")

┌─────────────────┬────────┐
│ parcels_on_deed │ deeds  │
│      int64      │ int64  │
├─────────────────┼────────┤
│               1 │ 434052 │
│               2 │  18955 │
│               3 │   3276 │
│               4 │   1226 │
│               5 │    621 │
│               6 │    311 │
│               7 │    182 │
│               8 │    104 │
│               9 │     94 │
│              10 │     61 │
│              11 │     45 │
│              12 │     35 │
│              13 │     30 │
│              14 │     17 │
│              15 │     33 │
└─────────────────┴────────┘
  15 rows        2 columns



`parcels_on_deed` = how many different parcels share one deed reference. `deeds` = how many deed references have that many parcels.

**What this shows:**
- 434,052 book + page pairs have 1 parcel and 18,955 have 2, and the numbers drop quickly after that. (This table still counts zero placeholders as deeds; the clean count is at the end of this section.)
- One parcel can have several records, and one deed can cover several parcels.
- So a deed can't be the key on its own: without `PARID`, deed + sale date leaves 44,692 rows that aren't unique (§2.5). Parcel + deed (+ sale date) is the candidate for one row, and placeholders are never used to group or join.

**Question:** what does a normal deed reference (book + page) look like: length, digits only, most common values?

In [75]:
# min() instead of any_value(): the example columns show the same values on every run
for col in ["DEEDBOOK", "DEEDPAGE"]:
    show(f"""
    SELECT '{col}' AS column_name, length({col}) AS len, count(*) AS rows,
           count(*) FILTER (WHERE regexp_matches({col}, '^[0-9]+$')) AS digits_only,
           min({col}) AS example,
           min({col}) FILTER (WHERE NOT regexp_matches({col}, '^[0-9]*$')) AS example_not_digits
    FROM s GROUP BY 2 ORDER BY 2
    """)
show("SELECT DEEDBOOK, DEEDPAGE, count(*) AS rows, count(DISTINCT PARID) AS parcels FROM s GROUP BY 1, 2 ORDER BY rows DESC, DEEDBOOK, DEEDPAGE LIMIT 10")

┌─────────────┬───────┬────────┬─────────────┬──────────┬────────────────────┐
│ column_name │  len  │  rows  │ digits_only │ example  │ example_not_digits │
│   varchar   │ int64 │ int64  │    int64    │ varchar  │      varchar       │
├─────────────┼───────┼────────┼─────────────┼──────────┼────────────────────┤
│ DEEDBOOK    │     0 │    567 │           0 │          │ NULL               │
│ DEEDBOOK    │     1 │   1114 │        1114 │ 0        │ NULL               │
│ DEEDBOOK    │     2 │    861 │         856 │ 00       │ DE                 │
│ DEEDBOOK    │     3 │    302 │         301 │ 000      │ T13                │
│ DEEDBOOK    │     4 │   5317 │         232 │ 0000     │ DT14               │
│ DEEDBOOK    │     5 │ 498676 │      498666 │ 00000    │ 1988Z              │
│ DEEDBOOK    │     6 │    105 │          88 │  14795   │  14795             │
│ DEEDBOOK    │     7 │      9 │           4 │ 0000000  │ GD-2022            │
│ DEEDBOOK    │     8 │     26 │          17 │ 01292

For each length: how many rows, and how many are digits only. `example` = the smallest value of that length, `example_not_digits` = the smallest one that isn't only digits, so both show the same values on every run.

**What this shows:** a normal book is 5 digits (498,676 rows), and a normal page is 1–3 digits. The odd ones are blanks (length 0), short zero values like 0 or 00, and books with letters (length 4 is mostly TR13–TR19).

**Question:** which deed references are shared by unrelated sales?

In [76]:
show("""
SELECT DEEDBOOK, DEEDPAGE, count(*) AS rows, count(DISTINCT PARID) AS parcels,
       count(DISTINCT SALEDATE) AS sale_dates, count(DISTINCT PRICE) AS prices,
       count(DISTINCT MUNICODE) AS municipalities, count(DISTINCT file_year) AS years
FROM s GROUP BY 1, 2 ORDER BY sale_dates DESC, rows DESC LIMIT 25
""")

┌──────────┬──────────┬───────┬─────────┬────────────┬────────┬────────────────┬───────┐
│ DEEDBOOK │ DEEDPAGE │ rows  │ parcels │ sale_dates │ prices │ municipalities │ years │
│ varchar  │ varchar  │ int64 │  int64  │   int64    │ int64  │     int64      │ int64 │
├──────────┼──────────┼───────┼─────────┼────────────┼────────┼────────────────┼───────┤
│ 0        │ 0        │   958 │     957 │        717 │      9 │             27 │    15 │
│ 00       │ 00       │   737 │     736 │        589 │      6 │             26 │    15 │
│          │          │   546 │     545 │        301 │     17 │            102 │    13 │
│ 000      │ 000      │   270 │     270 │        238 │      5 │             19 │    15 │
│ 0        │ 00       │    92 │      92 │         82 │      3 │             20 │     5 │
│ 0000     │ 0000     │    88 │      88 │         78 │      3 │             15 │    12 │
│ 00       │ 0        │    61 │      61 │         48 │      2 │             15 │     5 │
│ 0        │ 000     

**What this shows:** a deed is signed on one date, so a reference that spans many sale dates, prices, municipalities and years isn't one document. Above, 0/0 spans 717 sale dates, 27 municipalities and 15 years. Almost every reference near the top is blank or zeros. Real deeds like 19703/461 appear further down, with at most a few dates.

**Question:** checking every deed reference: how many span more than one sale date, and how many of those are blank or only zeros?

In [77]:
show("""
SELECT CASE WHEN sale_dates = 1 THEN '1 date' WHEN sale_dates <= 3 THEN '2-3 dates' ELSE '4+ dates' END AS dates_per_deed_reference,
       count(*) AS deed_references, sum(rows) AS rows,
       count(*) FILTER (WHERE blank_or_zero) AS of_which_blank_or_zero
FROM (SELECT DEEDBOOK, DEEDPAGE, count(*) AS rows, count(DISTINCT SALEDATE) AS sale_dates,
             trim(DEEDBOOK, '0 ') = '' OR trim(DEEDPAGE, '0 ') = '' AS blank_or_zero
      FROM s GROUP BY 1, 2)
GROUP BY 1 ORDER BY 1
""")

┌──────────────────────────┬─────────────────┬────────┬────────────────────────┐
│ dates_per_deed_reference │ deed_references │  rows  │ of_which_blank_or_zero │
│         varchar          │      int64      │ int128 │         int64          │
├──────────────────────────┼─────────────────┼────────┼────────────────────────┤
│ 1 date                   │          458398 │ 501171 │                     29 │
│ 2-3 dates                │             835 │   2897 │                      6 │
│ 4+ dates                 │              18 │   2909 │                     17 │
└──────────────────────────┴─────────────────┴────────┴────────────────────────┘



**What this shows:** almost every real deed has one date, and nearly every reference spanning 4 or more dates is a placeholder.
- 458,398 references have 1 date. 835 span 2–3 dates, and 829 of those are real references (only 6 are placeholders). So "one deed reference = one document on one date" mostly holds, but not always.
- 17 of the 18 references spanning 4+ dates are placeholders.

How a placeholder is found: remove all zeros and spaces; if nothing is left, the value was blank or only zeros.

**Question:** how many rows and parcels share a placeholder deed reference (blank or only zeros)?

In [78]:
show("""
SELECT DEEDBOOK, DEEDPAGE, count(*) AS rows, count(DISTINCT PARID) AS parcels
FROM s WHERE trim(DEEDBOOK, '0 ') = '' OR trim(DEEDPAGE, '0 ') = ''
GROUP BY 1, 2 ORDER BY rows DESC
""")

┌──────────┬──────────┬───────┬─────────┐
│ DEEDBOOK │ DEEDPAGE │ rows  │ parcels │
│ varchar  │ varchar  │ int64 │  int64  │
├──────────┼──────────┼───────┼─────────┤
│ 0        │ 0        │   958 │     957 │
│ 00       │ 00       │   737 │     736 │
│          │          │   546 │     545 │
│ 000      │ 000      │   270 │     270 │
│ 0        │ 00       │    92 │      92 │
│ 0000     │ 0000     │    88 │      88 │
│ 00       │ 0        │    61 │      61 │
│ 0        │ 000      │    33 │      33 │
│ 00000    │ 00000    │    29 │      29 │
│          │ 0        │    17 │      17 │
│ 00       │ 000      │    16 │      16 │
│ 0        │ 0000     │    13 │      13 │
│ 000      │ 0        │     9 │       9 │
│ 00       │ 0000     │     8 │       8 │
│ 0        │          │     8 │       8 │
│ 0        │ 00000    │     7 │       7 │
│ TR19     │          │     6 │       6 │
│ 000      │ 00       │     5 │       5 │
│ 000000   │ 000000   │     3 │       3 │
│ ORD 731  │          │     3 │   

**Question:** where do the blank or zero deed references come from?

In [79]:
show("""
SELECT file_year, count(*) AS rows FROM s
WHERE trim(DEEDBOOK, '0 ') = '' OR trim(DEEDPAGE, '0 ') = '' GROUP BY 1 ORDER BY 1
""")
show("""
SELECT SALECODE, SALEDESC, INSTRTYP, INSTRTYPDESC, count(*) AS rows,
       count(*) FILTER (WHERE try_cast(PRICE AS BIGINT) <= 1) AS price_le_1
FROM s WHERE trim(DEEDBOOK, '0 ') = '' OR trim(DEEDPAGE, '0 ') = ''
GROUP BY ALL ORDER BY rows DESC LIMIT 15
""", width=260)

┌───────────┬───────┐
│ file_year │ rows  │
│   int32   │ int64 │
├───────────┼───────┤
│      2012 │   223 │
│      2013 │   350 │
│      2014 │   191 │
│      2015 │   225 │
│      2016 │   174 │
│      2017 │   181 │
│      2018 │   252 │
│      2019 │   206 │
│      2020 │   222 │
│      2021 │   234 │
│      2022 │   165 │
│      2023 │   136 │
│      2024 │   155 │
│      2025 │   132 │
│      2026 │   102 │
└───────────┴───────┘
       15 rows     

┌──────────┬──────────────────────────────────┬──────────┬─────────────────────────┬───────┬────────────┐
│ SALECODE │             SALEDESC             │ INSTRTYP │      INSTRTYPDESC       │ rows  │ price_le_1 │
│ varchar  │             varchar              │ varchar  │         varchar         │ int64 │   int64    │
├──────────┼──────────────────────────────────┼──────────┼─────────────────────────┼───────┼────────────┤
│ N        │ NOT APPLICABLE                   │ MH       │ Mobile Home Sale        │  2169 │         57 │
│ 3      

**What this shows:**
- 102 to 350 rows in every year file, so it's a steady pattern, not a problem in one year.
- By sale code and instrument type (`price_le_1` = rows priced at $1 or less): most are Mobile Home Sale (MH), Memorandum of Lease (ML) and Certificate of Transfer (CT).

**Question:** the same count as the first table, but leaving out **all** placeholder references (blank or only zeros), with no row limit. Since zero placeholders no longer count as deeds, "1 parcel" drops from 434,052 to 434,029. The two lines of the second table add up to the number of real deed references: 434,029 + 25,170 = 459,199.

In [80]:
show("""
SELECT CASE WHEN n <= 10 THEN n::VARCHAR ELSE '11+' END AS parcels_on_deed,
       count(*) AS deeds, max(n) AS max_parcels
FROM (SELECT DEEDBOOK, DEEDPAGE, count(DISTINCT PARID) AS n FROM s
      WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> '' GROUP BY 1, 2)
GROUP BY 1 ORDER BY min(n)
""")
show("""
SELECT count(*) FILTER (WHERE n = 1) AS deeds_with_1_parcel, count(*) FILTER (WHERE n > 1) AS deeds_with_2plus_parcels, max(n) AS max_parcels
FROM (SELECT DEEDBOOK, DEEDPAGE, count(DISTINCT PARID) AS n FROM s
      WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> '' GROUP BY 1, 2)
""")

┌─────────────────┬────────┬─────────────┐
│ parcels_on_deed │ deeds  │ max_parcels │
│     varchar     │ int64  │    int64    │
├─────────────────┼────────┼─────────────┤
│ 1               │ 434029 │           1 │
│ 2               │  18952 │           2 │
│ 3               │   3272 │           3 │
│ 4               │   1226 │           4 │
│ 5               │    620 │           5 │
│ 6               │    310 │           6 │
│ 7               │    181 │           7 │
│ 8               │    102 │           8 │
│ 9               │     93 │           9 │
│ 10              │     61 │          10 │
│ 11+             │    353 │         173 │
└─────────────────┴────────┴─────────────┘
  11 rows                      3 columns

┌─────────────────────┬──────────────────────────┬─────────────┐
│ deeds_with_1_parcel │ deeds_with_2plus_parcels │ max_parcels │
│        int64        │          int64           │    int64    │
├─────────────────────┼──────────────────────────┼─────────────┤
│         

**What this shows:** 434,029 deeds cover 1 parcel, 23 fewer than in the first table because the zero placeholders are gone. 25,170 deeds cover 2 or more parcels, and the biggest covers 173. That's 434,029 + 25,170 = 459,199 real deed references, which are the ones silver groups on.

**Question:** when one parcel has more than one row on the same sale date, are those different deeds or the same deed repeated?

In [81]:
show("""
SELECT count(*) AS parcel_date_groups_with_2plus_rows,
       count(*) FILTER (WHERE deeds > 1)  AS different_deeds,
       count(*) FILTER (WHERE deeds = 1)  AS same_deed_repeated
FROM (SELECT PARID, SALEDATE, count(DISTINCT (DEEDBOOK, DEEDPAGE)) AS deeds FROM s GROUP BY 1, 2 HAVING count(*) > 1)
""")

┌────────────────────────────────────┬─────────────────┬────────────────────┐
│ parcel_date_groups_with_2plus_rows │ different_deeds │ same_deed_repeated │
│               int64                │      int64      │       int64        │
├────────────────────────────────────┼─────────────────┼────────────────────┤
│                               3096 │            3008 │                 88 │
└────────────────────────────────────┴─────────────────┴────────────────────┘



**What this shows:**
- 3,008 of the 3,096 parcel + date groups with 2+ rows have different deeds. So rows of one parcel on the same date are usually separate deeds, not repeated rows.
- The other 88 share one deed; whether they're real duplicates is checked next.

**Question:** are the 88 same-deed groups real duplicates? First check whether the rows in each group are the same in every column except `_id`, then look at them side by side.

In [82]:
# The 88 groups from above: same parcel, same sale date, same deed, 2+ rows
same_deed = """
    SELECT PARID, SALEDATE FROM s GROUP BY PARID, SALEDATE
    HAVING count(*) > 1 AND count(DISTINCT (DEEDBOOK, DEEDPAGE)) = 1
"""
# Every column except _id (row number) and file_year, listed explicitly
row_cols = ", ".join(f'"{c}"' for c in con.sql("SELECT * FROM s LIMIT 0").columns if c not in ("_id", "file_year"))

# Are the rows in each group identical apart from _id?
show(f"""
SELECT count(*) AS groups,
       count(*) FILTER (WHERE rows = 2) AS groups_of_2,
       count(*) FILTER (WHERE versions = 1) AS identical_except_id,
       count(*) FILTER (WHERE versions > 1) AS differ_in_some_column
FROM (SELECT PARID, SALEDATE, count(*) AS rows, count(DISTINCT ({row_cols})) AS versions
      FROM s JOIN ({same_deed}) USING (PARID, SALEDATE) GROUP BY PARID, SALEDATE)
""")

# The rows themselves, all columns (FULL_ADDRESS left out: it repeats the address columns), one group number per parcel + date
show(f"""
SELECT dense_rank() OVER (ORDER BY PARID, SALEDATE) AS grp, s.* EXCLUDE (FULL_ADDRESS)
FROM s JOIN ({same_deed}) USING (PARID, SALEDATE)
ORDER BY grp, file_year, _id
""", rows=200, width=1200)

┌────────┬─────────────┬─────────────────────┬───────────────────────┐
│ groups │ groups_of_2 │ identical_except_id │ differ_in_some_column │
│ int64  │    int64    │        int64        │         int64         │
├────────┼─────────────┼─────────────────────┼───────────────────────┤
│     88 │          88 │                   0 │                    88 │
└────────┴─────────────┴─────────────────────┴───────────────────────┘

┌───────┬───────────┬──────────┬──────────────────┬──────────────────┬──────────────────┬────────────────────┬───────────────────────┬────────────────────┬─────────────────────────┬────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬──────────────────────┬──────────┬─────────────────────────┬────────────┬────────────┬─────────┬──────────┬──────────┬──────────┬─────────────────────────────────────────┬──────────┬───────────────────┐
│  grp  │ file_year │   _id    │      PARID       │ PROPERTYHOUSENUM │ PROPERTYFRACTION │ PROPERTYADDRESSDIR │ 

**How to read:** the first table counts the groups, split into rows that are the same in every column except `_id` (`identical_except_id`) and rows that differ somewhere (`differ_in_some_column`). In the second table `grp` numbers each parcel + date group, so the rows of one group sit together.

**What this shows:** all 88 groups are pairs, and none is an exact copy: every pair differs in at least one column. The two rows always share the parcel, deed, sale date, address and file, so they describe the same sale recorded twice with different details. The sale code differs in 75 pairs, the price in 24, the instrument type in 7 and the record date in 3. In 59 pairs one row is coded 99 CORRECTIVE DEED / DUPLICATE SALE. (These counts come from query B in 03 §4.7.)

So these aren't exact duplicates but two records of one sale. Removing one needs a rule for which row to keep (for example, drop the row coded 99), not a plain DISTINCT. Query B in 03 §4.7 also shows that no pair has two valid-coded rows.

**Open question:** for these pairs, is the row coded 99 always the one to drop, and which row is right when neither is coded 99?

### 2.5 Candidate key

**Question:** which columns describe the sale and which only describe the parcel? If a parcel always has the same municipality, school district and address, those columns add nothing to a key.

In [83]:
show(" UNION ALL ".join(
    f"SELECT '{c}' AS column_name, count(*) FILTER (WHERE n > 1) AS parcels_with_2plus_values, count(*) AS parcels "
    f"FROM (SELECT PARID, count(DISTINCT \"{c}\") AS n FROM s GROUP BY PARID)"
    for c in ["MUNICODE", "SCHOOLCODE", "FULL_ADDRESS", "PROPERTYZIP"]))

┌──────────────┬───────────────────────────┬─────────┐
│ column_name  │ parcels_with_2plus_values │ parcels │
│   varchar    │           int64           │  int64  │
├──────────────┼───────────────────────────┼─────────┤
│ MUNICODE     │                         0 │  313906 │
│ SCHOOLCODE   │                         0 │  313906 │
│ FULL_ADDRESS │                         0 │  313906 │
│ PROPERTYZIP  │                         0 │  313906 │
└──────────────┴───────────────────────────┴─────────┘



**What this shows:** each parcel has one value for these columns across all files. They describe the parcel, not the sale, so they don't belong in a key. That leaves parcel, deed and sale date as key candidates. Whether the value is the one at the time of sale or today's isn't known. *Assumption:* 313,906 parcels with no change at all over 15 years suggests the source attaches the parcel's *current* values.

**Question:** which combination of columns identifies one row? Test each candidate for uniqueness.

In [84]:
candidates = {
    "PARID + DEEDBOOK + DEEDPAGE": "PARID, DEEDBOOK, DEEDPAGE",
    "+ SALEDATE": "PARID, DEEDBOOK, DEEDPAGE, SALEDATE",
    "+ SALEDATE + file_year": "PARID, DEEDBOOK, DEEDPAGE, SALEDATE, file_year",
}
show(" UNION ALL ".join(
    f"SELECT '{name}' AS candidate_key, count(*) AS rows, count(DISTINCT ({cols})) AS distinct_keys, "
    f"count(*) - count(DISTINCT ({cols})) AS rows_not_unique FROM s"
    for name, cols in candidates.items()))

┌─────────────────────────────┬────────┬───────────────┬─────────────────┐
│        candidate_key        │  rows  │ distinct_keys │ rows_not_unique │
│           varchar           │ int64  │     int64     │      int64      │
├─────────────────────────────┼────────┼───────────────┼─────────────────┤
│ PARID + DEEDBOOK + DEEDPAGE │ 506977 │        506847 │             130 │
│ + SALEDATE                  │ 506977 │        506887 │              90 │
│ + SALEDATE + file_year      │ 506977 │        506887 │              90 │
└─────────────────────────────┴────────┴───────────────┴─────────────────┘



**What this shows:** parcel + deed + sale date is the best candidate. Adding the file year doesn't help, so the repeats are inside the same file.

**Question:** does the result change if rows with blank or zero deed references are looked at separately from rows with a real deed?

In [85]:
show("""
SELECT CASE WHEN trim(DEEDBOOK, '0 ') = '' OR trim(DEEDPAGE, '0 ') = '' THEN 'blank/zero deed' ELSE 'real deed' END AS deed_reference,
       count(*) AS rows,
       count(*) - count(DISTINCT (PARID, DEEDBOOK, DEEDPAGE))           AS not_unique_parcel_deed,
       count(*) - count(DISTINCT (PARID, DEEDBOOK, DEEDPAGE, SALEDATE)) AS not_unique_plus_sale_date
FROM s GROUP BY 1 ORDER BY 1
""")

┌─────────────────┬────────┬────────────────────────┬───────────────────────────┐
│ deed_reference  │  rows  │ not_unique_parcel_deed │ not_unique_plus_sale_date │
│     varchar     │ int64  │         int64          │           int64           │
├─────────────────┼────────┼────────────────────────┼───────────────────────────┤
│ blank/zero deed │   2948 │                      3 │                         0 │
│ real deed       │ 504029 │                    127 │                        90 │
└─────────────────┴────────┴────────────────────────┴───────────────────────────┘



**What this shows:** the placeholder rows don't break the key. Once the sale date is added they're all unique, because parcel and date identify them. All 90 remaining repeats are on real deeds.

**Question:** is every column of the candidate key needed? (drop one column at a time; if uniqueness gets worse, the column is needed)

In [86]:
key_cols = ["PARID", "DEEDBOOK", "DEEDPAGE", "SALEDATE"]
show(" UNION ALL ".join(
    [f"SELECT 'all of {', '.join(key_cols)}' AS key_tested, count(*) - count(DISTINCT ({', '.join(key_cols)})) AS rows_not_unique FROM s"] +
    [f"SELECT 'without {c}', count(*) - count(DISTINCT ({', '.join(k for k in key_cols if k != c)})) FROM s" for c in key_cols]))

┌────────────────────────────────────────────┬─────────────────┐
│                 key_tested                 │ rows_not_unique │
│                  varchar                   │      int64      │
├────────────────────────────────────────────┼─────────────────┤
│ all of PARID, DEEDBOOK, DEEDPAGE, SALEDATE │              90 │
│ without PARID                              │           44692 │
│ without DEEDBOOK                           │             105 │
│ without DEEDPAGE                           │            2217 │
│ without SALEDATE                           │             130 │
└────────────────────────────────────────────┴─────────────────┘



**What this shows:** removing any column makes uniqueness worse, so all four are needed. Without the parcel, one deed covers many parcels, which gives 44,692 repeats. That's why a deed alone can't be the key.

**Question:** are the key columns always filled? A row with a blank or placeholder key can't be identified.

In [87]:
show(" UNION ALL ".join(
    f"SELECT '{c}' AS key_column, count(*) FILTER (WHERE trim({c}) = '') AS blank, "
    f"count(*) FILTER (WHERE trim({c}) <> '' AND trim({c}, '0 ') = '') AS only_zeros, count(*) AS rows FROM s"
    for c in key_cols))

┌────────────┬───────┬────────────┬────────┐
│ key_column │ blank │ only_zeros │  rows  │
│  varchar   │ int64 │   int64    │ int64  │
├────────────┼───────┼────────────┼────────┤
│ PARID      │     0 │          0 │ 506977 │
│ DEEDBOOK   │   567 │       2353 │ 506977 │
│ DEEDPAGE   │   579 │       2366 │ 506977 │
│ SALEDATE   │     0 │          0 │ 506977 │
└────────────┴───────┴────────────┴────────┘



**What this shows:** PARID and SALEDATE are always filled, so the pipeline can check them as NOT NULL. The deed columns can be empty, and placeholder rows have to be kept and treated as unknown, not rejected.

**Question:** for rows that repeat on PARID + deed + sale date: which columns differ inside each group?

In [88]:
key = "PARID, DEEDBOOK, DEEDPAGE, SALEDATE"
other = [c for c in con.sql("SELECT * FROM s LIMIT 0").columns if c not in ("PARID", "DEEDBOOK", "DEEDPAGE", "SALEDATE", "file_year")]
show(" UNION ALL ".join(
    f"SELECT '{c}' AS column_name, count(*) FILTER (WHERE n > 1) AS groups_where_it_differs "
    f"FROM (SELECT count(DISTINCT \"{c}\") AS n FROM s GROUP BY {key} HAVING count(*) > 1)"
    for c in other) + " ORDER BY groups_where_it_differs DESC")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬─────────────────────────┐
│       column_name       │ groups_where_it_differs │
│         varchar         │          int64          │
├─────────────────────────┼─────────────────────────┤
│ _id                     │                      90 │
│ SALECODE                │                      77 │
│ SALEDESC                │                      77 │
│ PRICE                   │                      25 │
│ INSTRTYP                │                       7 │
│ INSTRTYPDESC            │                       7 │
│ RECORDDATE              │                       3 │
│ PROPERTYSTATE           │                       0 │
│ SCHOOLDESC              │                       0 │
│ SCHOOLCODE              │                       0 │
│ PROPERTYFRACTION        │                       0 │
│ PROPERTYUNITNO          │                       0 │
│ PROPERTYADDRESSSUF      │                       0 │
│ MUNICODE                │                       0 │
│ PROPERTYCITY            │ 

In [89]:
# The repeated rows themselves
show("""
SELECT file_year, PARID, DEEDBOOK, DEEDPAGE, SALEDATE, RECORDDATE, PRICE, SALECODE, SALEDESC, INSTRTYP
FROM s
WHERE (PARID, DEEDBOOK, DEEDPAGE, SALEDATE) IN
      (SELECT (PARID, DEEDBOOK, DEEDPAGE, SALEDATE) FROM s GROUP BY PARID, DEEDBOOK, DEEDPAGE, SALEDATE HAVING count(*) > 1)
ORDER BY PARID, DEEDBOOK, DEEDPAGE, SALEDATE
LIMIT 40
""", width=300)

┌───────────┬──────────────────┬──────────┬──────────┬────────────┬────────────┬─────────┬──────────┬────────────────────────────────────────┬──────────┐
│ file_year │      PARID       │ DEEDBOOK │ DEEDPAGE │  SALEDATE  │ RECORDDATE │  PRICE  │ SALECODE │                SALEDESC                │ INSTRTYP │
│   int32   │     varchar      │ varchar  │ varchar  │  varchar   │  varchar   │ varchar │ varchar  │                varchar                 │ varchar  │
├───────────┼──────────────────┼──────────┼──────────┼────────────┼────────────┼─────────┼──────────┼────────────────────────────────────────┼──────────┤
│      2019 │ 0007B00150000000 │ 17860    │ 343      │ 2019-11-26 │ 2019-12-05 │ 50000   │ 99       │ CORRECTIVE DEED / DUPLICATE SALE       │ DE       │
│      2019 │ 0007B00150000000 │ 17860    │ 343      │ 2019-11-26 │ 2019-12-05 │ 270000  │ H        │ MULTI-PARCEL SALE                      │ SW       │
│      2015 │ 0011A00237000000 │ TR15     │ 00002    │ 2015-01-06 │ 2015-01-

**What this shows:** in the 90 repeated groups (180 rows) the sale code differs in 77, the price in 25, the instrument type in 7 and the record date in 3. The address, municipality and school columns never differ. (`_id` differs in all 90 only because it's a row number.) Each group is two records of one parcel on one deed and date with conflicting details, so there's no exact copy to remove. Silver keeps both rows and flags them.

**Question:** does the key survive when the source corrects data? In each recorded change (`meta.change_log`), a key that was removed and added back means the same row was corrected. Which columns changed for those keys?

In [90]:
changes = con.execute("SELECT year, diff_table FROM meta.change_log ORDER BY detected_at").fetchall()
if not changes:
    print("no content changes recorded yet")
else:
    k = ", ".join(key_cols)
    show(" UNION ALL ".join(
        f"SELECT {y} AS year, '{t}' AS diff_table, "
        f"(SELECT count(*) FROM {t} WHERE _change = 'added') AS rows_added, "
        f"(SELECT count(*) FROM {t} WHERE _change = 'removed') AS rows_removed, "
        f"(SELECT count(*) FROM (SELECT {k} FROM {t} WHERE _change = 'removed' INTERSECT SELECT {k} FROM {t} WHERE _change = 'added')) AS removed_keys_added_back"
        for y, t in changes))
    for y, t in changes:
        cols = [c for c in con.sql(f"SELECT * FROM {t} LIMIT 0").columns if c not in key_cols + ["_change"]]
        exprs = ", ".join(f'count(*) FILTER (WHERE r."{c}" IS DISTINCT FROM a."{c}") AS "{c}"' for c in cols)
        print(f"{t}: columns that changed for keys removed and added back")
        show(f"""
        SELECT column_name, keys_where_changed FROM (
            UNPIVOT (SELECT {exprs} FROM (SELECT * FROM {t} WHERE _change = 'removed') r
                     JOIN (SELECT * FROM {t} WHERE _change = 'added') a USING ({k}))
            ON COLUMNS(*) INTO NAME column_name VALUE keys_where_changed)
        WHERE keys_where_changed > 0 ORDER BY keys_where_changed DESC
        """)

┌───────┬──────────────────────┬────────────┬──────────────┬─────────────────────────┐
│ year  │      diff_table      │ rows_added │ rows_removed │ removed_keys_added_back │
│ int32 │       varchar        │   int64    │    int64     │          int64          │
├───────┼──────────────────────┼────────────┼──────────────┼─────────────────────────┤
│  2013 │ meta.diff_2013_v1_v2 │        181 │            0 │                       0 │
│  2018 │ meta.diff_2018_v1_v2 │        378 │            0 │                       0 │
│  2026 │ meta.diff_2026_v1_v2 │        139 │           28 │                      28 │
│  2017 │ meta.diff_2017_v1_v2 │          1 │            0 │                       0 │
│  2023 │ meta.diff_2023_v1_v2 │          1 │            0 │                       0 │
│  2025 │ meta.diff_2025_v1_v2 │          3 │            0 │                       0 │
│  2026 │ meta.diff_2026_v2_v3 │        502 │          206 │                     205 │
│  2022 │ meta.diff_2022_v1_v2 │          1

**What this shows:** of the keys removed in a correction, 28 of 28, 205 of 206 and 168 of 169 came back with the same key, and only SALECODE (400 times) and once PRICE changed. The 2 keys that didn't come back are real deletions or changed keys. So the key follows a record through the source's corrections, which `_id` can't do. The exception is the 90 repeated groups, where a key match is ambiguous.

**Open questions:** for the repeated groups, is the row coded 99 always the duplicate to drop? Are the address, municipality and school district the parcel's values today or at the time of the sale?